*Why this dataset?* 
If I had to work on a global challenge over the course of my career, it would be waste. In India, I briefly studied a women-led waste management system in a town called Ambikapur [image below]. I had the chance to visit many recycling facilities and dumping yards which made me more aware of the sheer scale of waste we're dealing with as humanity.  



On a personal level, I struggle with managing my own waste and learning more about the waste management system in New York has been part of my learning agenda and this was a great place to start. 


*What research questions emerged as I worked on this data?*
1. When and where can *I* drop off my food waste?
2. How do the boroughs differ in terms of food scrap collection and composting? And...why?





In [2]:
#Starting of simple. We import the pandas library and read the csv file into a dataframe!
import pandas as pd
df = pd.read_csv('nycwaste.csv')

In [7]:
#let's take a look at what the data looks like. We use the head() function to see the first 5 rows. 
df.head()

,Borough,NTAName,SiteName,SiteAddr,Hosted_By,Open_Month,Day_Hours,Notes,Website,BoroCD,...,Object ID,Location Point,App Android,App iOS,Assembly District,Congress District,DSNY District,DSNY Section,DSNY Zone,Senate District
0,Brooklyn,Bay Ridge,4th Avenue Presbyterian Church,"6753 4th Avenue, Brooklyn, NY 11220",4th Avenue Presbyterian Church,Year Round,Every day (Start Time: Dawn - End Time: Dusk),"No meat, bones, or dairy.",NaN,310,...,76179,POINT (-74.022767 40.635514),NaN,NaN,51,10,BKS10,BKS101,BKS,17
1,Manhattan,East Midtown-Turtle Bay,Dag Hammarskjold Plaza Greenmarket,E 47th St & 2nd Ave,GrowNYC,Year Round,Wednesday (Start Time: 8:00 AM - End Time: 12...,NaN,grownyc.org/compost,106,...,76039,POINT (-73.969036 40.752606),NaN,NaN,74,12,MN06,MN063,MN,28
2,Manhattan,Hell's Kitchen,Hudson River Park's Pier 84 at W. 44th St.,Pier 84 at W. 44th St. near dog park,Staff at Hudson River Park,Year Round,Every day (Start Time: 7:00 AM - End Time: 7:...,NaN,https://hudsonriverpark.org/the-park/sustainab...,104,...,76007,POINT (-74.00025 40.76346),NaN,NaN,67,12,MN04,MN043,MN,47
3,Manhattan,East Midtown-Turtle Bay,58th Street Library FSDO,127 East 58th Street,GrowNYC,Year Round,Wednesdays (Start Time: 7:30 AM - End Time: 1...,NaN,grownyc.org/compost,105,...,76000,POINT (-73.9693 40.76198),NaN,NaN,73,12,MN05,MN052,MN,28
4,Manhattan,Tribeca-Civic Center,Tribeca Greenmarket,Greenwich St. & Duane St,GrowNYC,Year Round,Saturday (Start Time: 8:00 AM - End Time: 1:0...,NaN,grownyc.org/compost,101,...,75912,POINT (-74.010793 40.717424),NaN,NaN,66,10,MN01,MN013,MN,27


In [8]:
#let's dig a bit deeper to see the data types and how much we would need to clean this datatset.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 591 entries, 0 to 590
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Borough             591 non-null    str    
 1   NTAName             591 non-null    str    
 2   SiteName            591 non-null    str    
 3   SiteAddr            201 non-null    str    
 4   Hosted_By           585 non-null    str    
 5   Open_Month          591 non-null    str    
 6   Day_Hours           591 non-null    str    
 7   Notes               421 non-null    str    
 8   Website             561 non-null    str    
 9   BoroCD              591 non-null    int64  
 10  CouncilDis          591 non-null    int64  
 11  ct2010              591 non-null    int64  
 12  BBL                 10 non-null     float64
 13  BIN                 10 non-null     float64
 14  Latitude            591 non-null    float64
 15  Longitude           591 non-null    float64
 16  PolicePrec         

In [ ]:
boroughs_count= df["Borough"].value_counts()
boroughs

Borough
Manhattan        251
Brooklyn         171
Queens            75
Bronx             75
Staten Island     19
Name: count, dtype: int64

In [11]:
percent_scrap_location = (df["Borough"].value_counts(normalize=True) * 100).round(1)
percent_scrap_location

Borough
Manhattan        42.5
Brooklyn         28.9
Queens           12.7
Bronx            12.7
Staten Island     3.2
Name: proportion, dtype: float64

I want to compare the *number of foodscrap drop-off locations* in each borough with their *respective populations*.

![Population in NYC boroughs](images/population.png)

In [10]:
#Let's do some rough calculations to understand the population per borough

population = pd.DataFrame({
    "Borough": ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"],
    "population": [1406332, 2653963, 1664862, 2358182, 501290]
})

pop_percent= (population["population"] / population["population"].sum() * 100).round(1)
pop_percent_df= pd.DataFrame({"Borough": population["Borough"], "population_percent": pop_percent})

pop_percent_df

,Borough,population_percent
0,Bronx,16.4
1,Brooklyn,30.9
2,Manhattan,19.4
3,Queens,27.5
4,Staten Island,5.8


In [26]:
# tabular comparison of the population percentage and the scrap location

# convert the series to a dataframe so it can be merged on Borough
percent_scrap_location_df = percent_scrap_location.reset_index()
percent_scrap_location_df.columns = ["Borough", "Percentage of total foodscrap locations in NYC"]

combined_df = pd.merge(
    pop_percent_df.rename(columns={"population_percent": "Percentage of total NYC Population"}),
    percent_scrap_location_df,
    on="Borough",
    how="outer"
)

print(combined_df.to_string(index=False))


      Borough  Percentage of total NYC Population  Percentage of total foodscrap locations in NYC
        Bronx                                16.4                                            12.7
     Brooklyn                                30.9                                            28.9
    Manhattan                                19.4                                            42.5
       Queens                                27.5                                            12.7
Staten Island                                 5.8                                             3.2


In [38]:
pop_col = "Percentage of total NYC Population"
food_col = "Percentage of total foodscrap locations in NYC"

print("NYC population vs food scrap locations, by borough")
print("🟦 = % of population    🟩 = % of food scrap locations")
print("-" * 70)

for borough, pop, food in zip(combined_df["Borough"], combined_df[pop_col], combined_df[food_col]):
    print(f"{borough:<14} Population  {'🟦' * round(pop)} {pop}%")
    print(f"{'':<14} Food scraps {'🟩' * round(food)} {food}%")
    print()

NYC population vs food scrap locations, by borough
🟦 = % of population    🟩 = % of food scrap locations
----------------------------------------------------------------------
Bronx          Population  🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦 16.4%
               Food scraps 🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩 12.7%

Brooklyn       Population  🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦 30.9%
               Food scraps 🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩 28.9%

Manhattan      Population  🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦 19.4%
               Food scraps 🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩 42.5%

Queens         Population  🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦 27.5%
               Food scraps 🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩 12.7%

Staten Island  Population  🟦🟦🟦🟦🟦🟦 5.8%
               Food scraps 🟩🟩🟩 3.2%



In [33]:
councils = df.groupby("Borough")["CouncilDis"].value_counts()
councils

Borough        CouncilDis
Bronx          15            15
               14            11
               16            11
               11            11
               18            10
               13             8
               17             5
               8              4
Brooklyn       36            44
               37            32
               35            26
               34            14
               41            12
               42             9
               39             9
               38             7
               33             7
               40             4
               45             3
               47             3
               46             1
Manhattan      3             40
               1             33
               2             31
               6             30
               5             24
               4             22
               7             21
               9             18
               10            17
              

**SECTION 2: What can we do with the missing values?**

In [9]:
#Looks like I'm in a bit of luck. Most columns seem to have 591 non-null values. I'm interested in checking what "actually" is missing.
print(df.isnull().sum())

Borough                 0
NTAName                 0
SiteName                0
SiteAddr              390
Hosted_By               6
Open_Month              0
Day_Hours               0
Notes                 170
Website                30
BoroCD                  0
CouncilDis              0
ct2010                  0
BBL                   581
BIN                   581
Latitude                0
Longitude               0
PolicePrec              0
Object ID               0
Location Point          0
App Android           201
App iOS               201
 Assembly District      0
 Congress District      0
DSNY District           2
 DSNY Section           8
DSNY Zone               0
Senate District         0
dtype: int64


 Is there a way for us to fix this otherwise close to perfect dataset? I want to start with variables with the smallest missing values. A few stand out to me: 
 1) Hosted_By
 2) Website
 3) DSNY District
 4) DSNY Section


In [39]:
hosts = df["Hosted_By"].value_counts().sort_values(ascending=False)
hosts

Hosted_By
Department of Sanitation                            390
GrowNYC                                              46
Staff at Hudson River Park                           10
NYC Compost Project Hosted by LES Ecology Center     10
Astoria Pug                                           8
                                                   ... 
Spuyten Duyvil PreSchool                              1
Volunteers at Drew Gardens                            1
Moffat St Community Garden                            1
Members at East 4th Street Community Garden           1
EL Garden                                             1
Name: count, Length: 110, dtype: int64

In [3]:
#starting with investigating the "Hosted_By" column. 
missing_hosts = df.loc[df["Hosted_By"].isnull(), ["Borough", "SiteName"]]
missing_hosts

,Borough,SiteName
65,Queens,McIntosh Neighborhood Assoc.
134,Queens,Far Rock Urban Agro-Education Center
269,Manhattan,West Side Community Garden
413,Queens,Terraza 7
446,Bronx,Rainbow Garden of Health
576,Bronx,Francis Martin Library


In these 6 sites, we don't know the organization hosting the composting bins from the dataset. I wonder why? Given that this is a relatively small number, I think a quick google search can give me some answers. 

1. **McIntosh Neighborhood Assoc** is not a DSNY run site and is run  by the Mcintosh Neighborhood association itself. 
2. **Far Rock Urban Agro-education Center** is hosted by NYC Parks GreenThumb as part of the city’s community garden network, with the Trumbull Center for Agricultural and Horticultural Education (TCAH). 
3. **The West Side Community Garden’s** food scrap (compost) drop‑off is organized and operated by the Garden itself, with volunteers on site to assist participants.
4. Food scrap drop‑off at **Terraza 7** is not a standalone program — it is part of New York City’s Department of Sanitation (DSNY) food scrap collection system. 
5. The **Rainbow Garden of Life and Health** is a GreenThumb community garden hosted by NYC Parks, and food scrap collection there is handled by the NYC Department of Sanitation (DSNY)
6. **Francis Martin Library** is hosted by the New York Public Library (NYPL) in partnership with The Botanical Gardens

Let's update these missing values based on this information.

In [57]:
#I refer to the location numbers I found in the table before to recheck the values. 
df.loc[[65, 134, 269, 413, 446, 576], ["SiteName", "Hosted_By"]]

,SiteName,Hosted_By
65,McIntosh Neighborhood Assoc.,McIntosh Neighborhood Association
134,Far Rock Urban Agro-Education Center,NYC Parks GreenThumb
269,West Side Community Garden,West Side Community Garden
413,Terraza 7,Department of Sanitation
446,Rainbow Garden of Health,Department of Sanitation
576,Francis Martin Library,New York Public Library


In [56]:
#using the locations above, I am changing the names of the hosts to remove missing values. 
df.loc[65,"Hosted_By"] = "McIntosh Neighborhood Association"
df.loc[134,"Hosted_By"] = "NYC Parks GreenThumb"
df.loc[269, "Hosted_By"] = "West Side Community Garden"
df.loc[413, "Hosted_By"] = "Department of Sanitation"
df.loc[446, "Hosted_By"] = "Department of Sanitation"
df.loc[576, "Hosted_By"] = "New York Public Library"